In [1]:
!pip install gymnasium[atari] gymnasium[accept-rom-license] ale-py -q

In [2]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
import psutil

In [3]:
import numpy as np
import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing, FrameStackObservation
import ale_py


gym.register_envs(ale_py)

def make_env(env_name="ALE/Pong-v5"):
    env = gym.make(env_name, render_mode=None, frameskip=1)
    env = AtariPreprocessing(env, scale_obs=False)  
    env = FrameStackObservation(env, 4)
    return env

In [4]:

env = make_env()
obs, info = env.reset()

print(f"Observation shape: {obs.shape}")
print(f"Observation dtype: {obs.dtype}")
print(f"Obs min/max: {obs.min():.3f} / {obs.max():.3f}")
print(f"Number of actions: {env.action_space.n}")
print(f"Action meanings: {env.unwrapped.get_action_meanings()}")

env.close()

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Observation shape: (4, 84, 84)
Observation dtype: uint8
Obs min/max: 52.000 / 236.000
Number of actions: 6
Action meanings: ['NOOP', 'FIRE', 'RIGHT', 'LEFT', 'RIGHTFIRE', 'LEFTFIRE']


In [5]:
import random
from collections import deque
import torch

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        
        self.buffer.append((
            np.array(state, dtype=np.uint8),
            action,
            reward,
            np.array(next_state, dtype=np.uint8),
            done
        ))

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in indices]
        
        states      = np.array([b[0] for b in batch], dtype=np.float32) / 255.0
        actions     = np.array([b[1] for b in batch], dtype=np.int64)
        rewards     = np.array([b[2] for b in batch], dtype=np.float32)
        next_states = np.array([b[3] for b in batch], dtype=np.float32) / 255.0
        dones       = np.array([b[4] for b in batch], dtype=np.float32)
    
        return (
            torch.from_numpy(states),
            torch.from_numpy(actions),
            torch.from_numpy(rewards),
            torch.from_numpy(next_states),
            torch.from_numpy(dones),
        )
    def __len__(self):
        return len(self.buffer)

In [6]:
import torch.nn as nn

class DQN(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels=4,  out_channels=32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1),
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(3136, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )
    def forward(self, x):
        
        x = self.conv(x)       
        x = x.view(x.size(0), -1)  
        x = self.fc(x)       
        return x
        

In [7]:
class Agent:
    def __init__(self, env, network, buffer, device,
                 epsilon_start=1.0,
                 epsilon_end=0.01,
                 epsilon_decay_steps=100_000):

        self.env = env
        self.network = network
        self.buffer = buffer
        self.device = device

       
        self.epsilon_start = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay_steps = epsilon_decay_steps

        
        self.total_steps = 0
        self.episode_reward = 0.0
        self.completed_episodes = 0

        
        self.state, _ = self.env.reset()

    def epsilon(self):
        
        progress = min(self.total_steps / self.epsilon_decay_steps, 1.0)
        return self.epsilon_start + progress * (self.epsilon_end - self.epsilon_start)

    @torch.no_grad()
    def select_action(self, state):
        if random.random() < self.epsilon():
            
            return self.env.action_space.sample()
        else:
            
            state_t = torch.tensor(np.array(state), dtype=torch.float32).unsqueeze(0).to(self.device) / 255.0
            q_values = self.network(state_t)       
            return q_values.argmax(dim=1).item()   

    def step(self):
        
        action = self.select_action(self.state)

       
        next_state, reward, terminated, truncated, _ = self.env.step(action)
        done = terminated or truncated

        
        self.buffer.push(self.state, action, reward, next_state, done)

       
        self.total_steps += 1
        self.episode_reward += reward
        self.state = next_state

       
        episode_info = None
        if done:
            episode_info = {
                "episode": self.completed_episodes,
                "reward": self.episode_reward,
                "steps": self.total_steps,
                "epsilon": self.epsilon()
            }
            self.episode_reward = 0.0
            self.completed_episodes += 1
            self.state, _ = self.env.reset()

        return episode_info

In [8]:
import torch
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name}, {props.total_memory / 1e9:.1f} GB VRAM")
print(f"Currently allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Currently reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

GPU count: 2
GPU 0: Tesla T4, 15.6 GB VRAM
GPU 1: Tesla T4, 15.6 GB VRAM
Currently allocated: 0.00 GB
Currently reserved:  0.00 GB


In [9]:
import torch.optim as optim

def train(env_name="ALE/Pong-v5",
          buffer_capacity=25000,
          batch_size=32,
          min_buffer_size=5000,
          total_steps=2000000,
          learning_rate=1e-4,
          gamma=0.99,
          target_update_freq=1_000,
          epsilon_start=1.0,
          epsilon_end=0.01,
          epsilon_decay_steps=100_000,
          log_every=10):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch.cuda.empty_cache()
    print(f"Training on: {device}")

    env = make_env(env_name)
    n_actions = env.action_space.n

    main_net = DQN(n_actions).to(device)
    target_net = DQN(n_actions).to(device)

    print(f"Using single GPU: {torch.cuda.get_device_name(0)}")

    target_net.load_state_dict(main_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(main_net.parameters(), lr=learning_rate)
    buffer = ReplayBuffer(buffer_capacity)
    agent = Agent(env, main_net, buffer, device,
                  epsilon_start, epsilon_end, epsilon_decay_steps)

    episode_rewards = []
    loss_fn = nn.MSELoss()

    for step in range(total_steps):

        episode_info = agent.step()
        
        if episode_info:
            episode_rewards.append(episode_info["reward"])
            if episode_info["episode"] % log_every == 0:
                mean_reward = np.mean(episode_rewards[-100:])
                print(f"Episode: {episode_info['episode']:4d} | "
                      f"Reward: {episode_info['reward']:6.1f} | "
                      f"Mean(100): {mean_reward:6.1f} | "
                      f"Epsilon: {episode_info['epsilon']:.3f} | "
                      f"Steps: {episode_info['steps']:7d}")

        if len(buffer) < min_buffer_size:
            continue

        states, actions, rewards, next_states, dones = buffer.sample(batch_size)

        try:
            states      = states.to(device, non_blocking=True)
            actions     = actions.to(device, non_blocking=True)
            rewards     = rewards.to(device, non_blocking=True)
            next_states = next_states.to(device, non_blocking=True)
            dones       = dones.to(device, non_blocking=True)
        except RuntimeError as e:
            print(f"OOM at step {step}")
            print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
            print(f"Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
            raise e
        q_values = main_net(states)
        q_values = q_values.gather(1, actions.unsqueeze(1))
        q_values = q_values.squeeze(1)

        with torch.no_grad():
            next_q_values = target_net(next_states).max(dim=1)[0]
            targets = (rewards + gamma * next_q_values * (1.0 - dones)).detach()

        loss = loss_fn(q_values, targets)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(main_net.parameters(), max_norm=10)
        optimizer.step()

        del states, actions, rewards, next_states, dones
        del q_values, next_q_values, targets, loss
        torch.cuda.synchronize()

        if step % target_update_freq == 0:
            target_net.load_state_dict(main_net.state_dict())
        if step % 10_000 == 0:
            torch.cuda.empty_cache()

    env.close()
    print("Training complete!")
    return main_net, episode_rewards

In [10]:
main_net, episode_rewards = train(
    buffer_capacity=150_000,
    batch_size=64,
    min_buffer_size=50_000,
    total_steps=2_000_000,
    learning_rate=1e-4,
    target_update_freq=2_000,
    epsilon_decay_steps=500_000,
)
torch.save(main_net.state_dict(), 'pong_dqn.pth')

Training on: cuda
Using single GPU: Tesla T4
Episode:    0 | Reward:  -21.0 | Mean(100):  -21.0 | Epsilon: 0.998 | Steps:     778
Episode:   10 | Reward:  -19.0 | Mean(100):  -20.6 | Epsilon: 0.981 | Steps:    9765
Episode:   20 | Reward:  -20.0 | Mean(100):  -20.6 | Epsilon: 0.962 | Steps:   19030
Episode:   30 | Reward:  -21.0 | Mean(100):  -20.5 | Epsilon: 0.943 | Steps:   28679
Episode:   40 | Reward:  -21.0 | Mean(100):  -20.5 | Epsilon: 0.925 | Steps:   38013
Episode:   50 | Reward:  -21.0 | Mean(100):  -20.5 | Epsilon: 0.907 | Steps:   47188
Episode:   60 | Reward:  -20.0 | Mean(100):  -20.5 | Epsilon: 0.889 | Steps:   56140
Episode:   70 | Reward:  -21.0 | Mean(100):  -20.5 | Epsilon: 0.870 | Steps:   65823
Episode:   80 | Reward:  -21.0 | Mean(100):  -20.5 | Epsilon: 0.853 | Steps:   74481
Episode:   90 | Reward:  -20.0 | Mean(100):  -20.4 | Epsilon: 0.833 | Steps:   84379
Episode:  100 | Reward:  -21.0 | Mean(100):  -20.4 | Epsilon: 0.812 | Steps:   94859
Episode:  110 | Rewa

In [11]:
import cv2
import numpy as np
import torch

def record_agent(net, env_name="ALE/Pong-v5", n_episodes=3, output_path="pong_agent.mp4", fps=30):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # need render_mode="rgb_array" to capture frames
    env = gym.make(env_name, render_mode="rgb_array", frameskip=1)
    env = AtariPreprocessing(env)
    env = FrameStackObservation(env, 4)

    # get frame size from a test render
    obs, _ = env.reset()
    raw_frame = env.render()
    height, width, _ = raw_frame.shape

    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    # unwrap DataParallel if needed so we can call the network directly
    model = net.module if hasattr(net, "module") else net
    model.eval()

    for episode in range(n_episodes):
        obs, _ = env.reset()
        total_reward = 0.0
        done = False

        while not done:
            # render and write frame
            frame = env.render()
            frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            writer.write(frame_bgr)

            # select best action — no exploration
            with torch.no_grad():
                state_t = torch.tensor(np.array(obs), dtype=torch.float32).unsqueeze(0).to(device) / 255.0
                action = model(state_t).argmax(dim=1).item()

            obs, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated

        print(f"Episode {episode + 1} | Reward: {total_reward:.1f}")

    writer.release()
    env.close()
    print(f"\nVideo saved to {output_path}")

record_agent(main_net, n_episodes=10)

[h264_v4l2m2m @ 0xb5529280] Could not find a valid device
[h264_v4l2m2m @ 0xb5529280] can't configure encoder
[ERROR:0@34057.790] global cap_ffmpeg_impl.hpp:3514 open Could not open codec h264_v4l2m2m, error: Unspecified error (-22)
[ERROR:0@34057.790] global cap_ffmpeg_impl.hpp:3531 open VIDEOIO/FFMPEG: Failed to initialize VideoWriter


Episode 1 | Reward: 15.0
Episode 2 | Reward: 10.0
Episode 3 | Reward: 9.0
Episode 4 | Reward: 17.0
Episode 5 | Reward: 10.0
Episode 6 | Reward: 15.0
Episode 7 | Reward: 13.0
Episode 8 | Reward: 7.0
Episode 9 | Reward: 13.0
Episode 10 | Reward: 11.0

Video saved to pong_agent.mp4


In [12]:

ram = psutil.virtual_memory()
print(f"Total RAM: {ram.total / 1e9:.1f} GB")
print(f"Available RAM: {ram.available / 1e9:.1f} GB")
print(f"Used RAM: {ram.used / 1e9:.1f} GB")

Total RAM: 33.7 GB
Available RAM: 22.9 GB
Used RAM: 10.3 GB
